# Coffee quality benchmark – training pipeline with MLflow

MLflow based version of
`coffee-quality-pricing-benchmark_training_pipeline.ipynb`. The pipeline
predicts the continuous `coffee_quality_score` target and stores every
artifact with MLflow in the `coffeePricePredictor/artifacts` folder:

* `artifacts/mlflow.db` – MLflow tracking and model registry (SQLite)
* `artifacts/mlruns/` – run and model artifacts

**Steps**

1. Configuration and MLflow setup
2. Data loading and validation
3. Feature preparation (removal of leakage columns)
4. Train/test split
5. Preprocessing (median/mode imputation, scaling, one-hot encoding)
6. Start of the parent MLflow run
7. Training of the candidate models – one nested MLflow run per model
   (RandomForest, XGBoost, LightGBM) with parameters, metrics and the
   logged model (preprocessor + regressor as one scikit-learn pipeline)
8. Selection of the best model (test R²) and logging of the comparison
9. Registration of the best model in the MLflow Model Registry with the
   `champion` alias
10. Verification of the registered model

The registered model is consumed by
`coffee-quality-pricing-benchmark_inference_pipeline_mlflow.ipynb`.

> The models are saved in `cloudpickle` format, because the default
> `skops` format of MLflow does not support LightGBM and XGBoost models.
> Load the models only from this trusted, local MLflow store.

In [1]:
"""Training pipeline for the coffee quality model with MLflow."""

import os
import time
from pathlib import Path
from typing import Any

import lightgbm as lgb
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import xgboost as xgb
from mlflow import MlflowClient
from mlflow.models import infer_signature
from mlflow.models.model import ModelInfo
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

2026/10/21 15:59:14 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\bordasm\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## 1. Configuration and MLflow setup

Paths are resolved relative to the `pipelines` folder, so the notebook
has to be run with `pipelines` as the working directory (default in VS
Code and Jupyter).

In [2]:
PIPELINE_DIR = Path.cwd()
PROJECT_ROOT = PIPELINE_DIR.parent
DATA_PATH = PROJECT_ROOT / "data" / "coffee_value_benchmark.csv"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
TRACKING_URI = "sqlite:///" + (ARTIFACTS_DIR / "mlflow.db").as_posix()
MLRUNS_DIR = ARTIFACTS_DIR / "mlruns"

EXPERIMENT_NAME = "coffee-quality-training"
REGISTERED_MODEL_NAME = "coffee-quality-model"
MODEL_ALIAS = "champion"
MODEL_ARTIFACT_NAME = "model"

TARGET_COLUMN = "coffee_quality_score"
LEAKAGE_COLUMNS = (
    "record_id",
    "lot_id",
    "quality_grade",
    "price_usd_per_kg",
)
TEST_SIZE = 0.2
RANDOM_STATE = 42
SELECTION_METRIC = "test_r2"

os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

print(f"Data path:    {DATA_PATH}")
print(f"Tracking URI: {TRACKING_URI}")
print(f"Artifacts:    {MLRUNS_DIR}")

Data path:    c:\D\Work\Ismeret\AI_ML_DL_TL\CUBIX_MLOPS_AIOPS\coffeePricePredictor\data\coffee_value_benchmark.csv
Tracking URI: sqlite:///c:/D/Work/Ismeret/AI_ML_DL_TL/CUBIX_MLOPS_AIOPS/coffeePricePredictor/artifacts/mlflow.db
Artifacts:    c:\D\Work\Ismeret\AI_ML_DL_TL\CUBIX_MLOPS_AIOPS\coffeePricePredictor\artifacts\mlruns


In [3]:
def setup_mlflow(experiment_name: str) -> str:
    """Configure the tracking store and return the experiment id."""
    ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
    mlflow.set_tracking_uri(TRACKING_URI)

    experiment = mlflow.get_experiment_by_name(experiment_name)
    if experiment is None:
        experiment_id = mlflow.create_experiment(
            experiment_name,
            artifact_location=(MLRUNS_DIR / experiment_name).as_uri(),
        )
    else:
        experiment_id = experiment.experiment_id
    mlflow.set_experiment(experiment_name)
    return experiment_id


experiment_id = setup_mlflow(EXPERIMENT_NAME)
client = MlflowClient()

print(f"Experiment: {EXPERIMENT_NAME} (id: {experiment_id})")

2026/10/21 15:59:17 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/21 15:59:17 INFO mlflow.store.db.utils: Updating database tables


Experiment: coffee-quality-training (id: 1)


## 2. Data loading and validation

In [4]:
def load_dataset(data_path: Path) -> pd.DataFrame:
    """Load the benchmark CSV file and validate that it is not empty."""
    if not data_path.exists():
        raise FileNotFoundError(f"Dataset not found: {data_path}")

    dataframe = pd.read_csv(data_path)
    if dataframe.empty:
        raise ValueError(f"Dataset is empty: {data_path}")
    return dataframe


def validate_dataset(
    dataframe: pd.DataFrame,
    target_column: str,
    leakage_columns: tuple[str, ...],
) -> None:
    """Check that the required columns and target values are present."""
    required_columns = {target_column, *leakage_columns}
    missing = required_columns - set(dataframe.columns)
    if missing:
        raise ValueError(
            f"Dataset is missing required columns: {sorted(missing)}"
        )
    if dataframe[target_column].isna().any():
        raise ValueError(
            f"Target column '{target_column}' contains missing values."
        )

In [5]:
raw_data = load_dataset(DATA_PATH)
validate_dataset(raw_data, TARGET_COLUMN, LEAKAGE_COLUMNS)

print(f"Dataset shape: {raw_data.shape}")
raw_data.head()

Dataset shape: (35700, 36)


,record_id,lot_id,harvest_season,harvest_calendar_year,harvest_month,country_of_origin,farm_type,farm_size_hectares,tree_age_years,altitude_mean_meters,...,lot_size_kg,roast_level,acidity_score,sweetness_score,body_score,aroma_score,flavor_score,coffee_quality_score,quality_grade,price_usd_per_kg
0,REC-0000001,LOT-0032936,2020/21,2020,10,Vietnam,Smallholder,0.58,28.0,1961.0,...,26433.0,Medium,7.95,7.30,8.45,7.35,7.37,78.01,Very Good,2.43
1,REC-0000002,LOT-0030952,2021/22,2022,1,Honduras,Smallholder,1.12,2.0,1257.0,...,25446.0,Medium,7.62,8.17,7.45,7.80,6.88,75.62,Good,4.06
2,REC-0000003,LOT-0018572,2020/21,2020,11,Vietnam,Cooperative,13.82,5.0,557.0,...,15707.0,Medium,8.47,8.83,8.13,7.70,8.10,72.41,Commercial,4.26
3,REC-0000004,LOT-0005318,2021/22,2021,10,Vietnam,Cooperative,9.96,22.0,652.0,...,2879.0,Medium,7.54,8.12,8.21,7.77,7.13,75.67,Good,3.60
4,REC-0000005,LOT-0006317,2021/22,2021,7,Brazil,Smallholder,1.90,11.0,971.0,...,9561.0,Medium,6.70,8.08,8.68,7.38,7.16,73.77,Good,4.16


## 3. Feature preparation

Identifier and outcome columns (`record_id`, `lot_id`, `quality_grade`,
`price_usd_per_kg`) are removed to avoid target leakage. Numeric
features are converted to `float64`, so the MLflow model signature
accepts missing values in every numeric column.

In [6]:
def prepare_features(
    dataframe: pd.DataFrame,
    target_column: str,
    leakage_columns: tuple[str, ...],
) -> tuple[pd.DataFrame, pd.Series]:
    """Split the data into features and target without leakage columns."""
    drop_columns = [*leakage_columns, target_column]
    features = dataframe.drop(columns=drop_columns, errors="ignore")
    numerical_columns = features.select_dtypes(include=["number"]).columns
    features[numerical_columns] = features[numerical_columns].astype(
        "float64"
    )
    target = dataframe[target_column].astype("float64")
    return features, target


features, target = prepare_features(raw_data, TARGET_COLUMN, LEAKAGE_COLUMNS)

print(f"Number of features: {features.shape[1]}")
target.describe()

Number of features: 31


count    35700.000000
mean        77.602334
std          4.915255
min         60.000000
25%         73.810000
50%         78.240000
75%         81.330000
max         94.180000
Name: coffee_quality_score, dtype: float64

## 4. Train/test split

In [7]:
x_train, x_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
)

print(f"Train rows: {len(x_train)}")
print(f"Test rows:  {len(x_test)}")

Train rows: 28560
Test rows:  7140


## 5. Preprocessing

* numeric columns: median imputation + standard scaling
* categorical columns: most frequent imputation + one-hot encoding

The preprocessor is fitted on the training split only.

In [8]:
def build_preprocessor(features: pd.DataFrame) -> ColumnTransformer:
    """Create a preprocessing pipeline for numeric and categorical data."""
    numerical_columns = features.select_dtypes(
        include=["number"]
    ).columns.tolist()
    categorical_columns = features.select_dtypes(
        exclude=["number"]
    ).columns.tolist()
    if not numerical_columns:
        raise ValueError("No numeric columns were found in the feature set.")

    numerical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "encoder",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )
    return ColumnTransformer(
        transformers=[
            ("num", numerical_pipeline, numerical_columns),
            ("cat", categorical_pipeline, categorical_columns),
        ]
    )


preprocessor = build_preprocessor(x_train)
x_train_processed = preprocessor.fit_transform(x_train)
x_test_processed = preprocessor.transform(x_test)

print(f"Processed train shape: {x_train_processed.shape}")
print(f"Processed test shape:  {x_test_processed.shape}")

Processed train shape: (28560, 61)
Processed test shape:  (7140, 61)


## 6. Start of the parent MLflow run

The parent run holds the pipeline configuration, the training dataset
reference, the model comparison and the selected best model. A run left
open by an earlier, interrupted execution is closed as `FAILED`.

In [9]:
if mlflow.active_run() is not None:
    mlflow.end_run(status="FAILED")

parent_run = mlflow.start_run(run_name="training-pipeline")
mlflow.set_tags(
    {
        "pipeline": "training",
        "source": "training_pipeline_mlflow.ipynb",
        "target_column": TARGET_COLUMN,
    }
)
mlflow.log_params(
    {
        "data_path": str(DATA_PATH),
        "target_column": TARGET_COLUMN,
        "leakage_columns": ",".join(LEAKAGE_COLUMNS),
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "selection_metric": SELECTION_METRIC,
        "n_rows": len(raw_data),
        "n_features": features.shape[1],
        "n_train_rows": len(x_train),
        "n_test_rows": len(x_test),
    }
)
mlflow.log_input(
    mlflow.data.from_pandas(
        raw_data,
        source=str(DATA_PATH),
        targets=TARGET_COLUMN,
        name="coffee_value_benchmark",
    ),
    context="training",
)

print(f"Parent run id: {parent_run.info.run_id}")

C:\Users\bordasm\.venv\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


Parent run id: c7a6fd63c70c4f53afe0c4e760ff5e78


C:\Users\bordasm\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


## 7. Training of the candidate models

Each candidate is trained in a nested run. The fitted preprocessor and
the regressor are logged together as one scikit-learn pipeline, so the
logged model accepts raw feature data.

In [10]:
def build_models(random_state: int) -> dict[str, Any]:
    """Define the candidate regression models of the benchmark study."""
    return {
        "RandomForest": RandomForestRegressor(
            n_estimators=200,
            max_depth=15,
            n_jobs=-1,
            random_state=random_state,
        ),
        "XGBoost": xgb.XGBRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            n_jobs=-1,
            random_state=random_state,
            verbosity=0,
        ),
        "LightGBM": lgb.LGBMRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            n_jobs=-1,
            random_state=random_state,
            verbose=-1,
        ),
    }


def evaluate_predictions(
    y_true: pd.Series,
    y_pred: np.ndarray,
    prefix: str,
) -> dict[str, float]:
    """Compute regression metrics for a prediction set."""
    return {
        f"{prefix}_rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        f"{prefix}_mae": float(mean_absolute_error(y_true, y_pred)),
        f"{prefix}_r2": float(r2_score(y_true, y_pred)),
    }


def train_and_log_model(
    model_name: str,
    model: Any,
    preprocessor: ColumnTransformer,
) -> tuple[dict[str, float], ModelInfo]:
    """Fit one candidate model and log it in a nested MLflow run."""
    with mlflow.start_run(run_name=model_name, nested=True):
        start = time.perf_counter()
        model.fit(x_train_processed, y_train)
        elapsed = time.perf_counter() - start

        metrics = {
            **evaluate_predictions(
                y_train, model.predict(x_train_processed), "train"
            ),
            **evaluate_predictions(
                y_test, model.predict(x_test_processed), "test"
            ),
            "fit_time_seconds": round(elapsed, 3),
        }
        mlflow.set_tags({"pipeline": "training", "model_name": model_name})
        mlflow.log_param("model_name", model_name)
        mlflow.log_params(
            {
                f"model__{key}": value
                for key, value in model.get_params().items()
            }
        )
        mlflow.log_metrics(metrics)

        model_pipeline = Pipeline(
            steps=[("preprocessor", preprocessor), ("model", model)]
        )
        input_example = x_train.head(5)
        model_info = mlflow.sklearn.log_model(
            model_pipeline,
            name=MODEL_ARTIFACT_NAME,
            serialization_format="cloudpickle",
            signature=infer_signature(
                input_example, model_pipeline.predict(input_example)
            ),
            input_example=input_example,
        )
    print(f"{model_name} trained and logged in {elapsed:.2f} s")
    return metrics, model_info

In [11]:
results: dict[str, dict[str, float]] = {}
model_infos: dict[str, ModelInfo] = {}
for model_name, model in build_models(RANDOM_STATE).items():
    results[model_name], model_infos[model_name] = train_and_log_model(
        model_name, model, preprocessor
    )

2026/10/21 15:59:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RandomForest trained and logged in 18.76 s


2026/10/21 16:00:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoost trained and logged in 1.75 s


2026/10/21 16:00:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


LightGBM trained and logged in 2.04 s


## 8. Selection of the best model

In [12]:
comparison = (
    pd.DataFrame(results)
    .T.rename_axis("model")
    .sort_values(SELECTION_METRIC, ascending=False)
)
best_model_name = str(comparison.index[0])
best_metrics = results[best_model_name]

mlflow.log_table(comparison.reset_index(), "model_comparison.json")
mlflow.log_dict(
    {
        "best_model": best_model_name,
        "selection_metric": SELECTION_METRIC,
        "best_model_metrics": best_metrics,
        "all_models": results,
        "feature_columns": features.columns.tolist(),
        "best_model_uri": model_infos[best_model_name].model_uri,
    },
    "model_summary.json",
)
mlflow.log_param("best_model", best_model_name)
mlflow.log_metrics(
    {f"best_{key}": value for key, value in best_metrics.items()}
)

print(f"Best model: {best_model_name}")
comparison

Best model: LightGBM


,train_rmse,train_mae,train_r2,test_rmse,test_mae,test_r2,fit_time_seconds
model,,,,,,,
LightGBM,1.475656,1.173199,0.909844,1.670481,1.323586,0.884598,2.044
XGBoost,1.385374,1.097875,0.920538,1.674202,1.324503,0.884083,1.754
RandomForest,0.888135,0.710752,0.967343,1.740039,1.374860,0.874787,18.764


## 9. Registration of the best model

The best model becomes a new version of the registered model and gets
the `champion` alias, which is used by the inference pipeline and the
REST API. The parent run is closed afterwards.

In [13]:
best_model_info = model_infos[best_model_name]
model_version = mlflow.register_model(
    best_model_info.model_uri,
    REGISTERED_MODEL_NAME,
    tags={
        "model_name": best_model_name,
        "training_run_id": parent_run.info.run_id,
        SELECTION_METRIC: f"{best_metrics[SELECTION_METRIC]:.6f}",
    },
)
client.set_registered_model_alias(
    REGISTERED_MODEL_NAME, MODEL_ALIAS, model_version.version
)
client.update_model_version(
    REGISTERED_MODEL_NAME,
    model_version.version,
    description=(
        f"{best_model_name} predicting {TARGET_COLUMN}; "
        f"{SELECTION_METRIC}={best_metrics[SELECTION_METRIC]:.4f}"
    ),
)
mlflow.set_tags(
    {
        "registered_model_name": REGISTERED_MODEL_NAME,
        "registered_model_version": model_version.version,
    }
)
mlflow.end_run()

print(f"Registered model: {REGISTERED_MODEL_NAME}")
print(f"Version:          {model_version.version}")
print(f"Alias:            {MODEL_ALIAS}")

Successfully registered model 'coffee-quality-model'.
Created version '1' of model 'coffee-quality-model'.


Registered model: coffee-quality-model
Version:          1
Alias:            champion


## 10. Verification of the registered model

The model is loaded back from the registry by its alias and the test
metrics are recomputed on the raw test features.

In [14]:
champion_uri = f"models:/{REGISTERED_MODEL_NAME}@{MODEL_ALIAS}"
champion_model = mlflow.sklearn.load_model(champion_uri)

reloaded_metrics = evaluate_predictions(
    y_test, champion_model.predict(x_test), "test"
)
for metric, value in reloaded_metrics.items():
    if not np.isclose(value, best_metrics[metric]):
        raise AssertionError(f"Metric mismatch after reload: {metric}")

print(f"Registered model verified: {champion_uri}")
pd.Series(reloaded_metrics)

Registered model verified: models:/coffee-quality-model@champion


test_rmse    1.670481
test_mae     1.323586
test_r2      0.884598
dtype: float64